# 02 - Exploração da camada Silver
## Tech Challenge Fase 3 — State of Data Brasil (2023, 2024, 2025-2026)

Este notebook lê os arquivos **Parquet** gerados pelo Glue Job `bronze_to_silver`
(camada Silver: `s3://.../Silver/state_of_data/`, particionada por `ano_pesquisa`)
e valida se a padronização e a limpeza saíram como esperado.

Como os dados já saem do Glue Job em Parquet, a leitura aqui usa `pandas` + `pyarrow`
(engine nativo do pandas para Parquet), lendo diretamente da estrutura particionada
— o mesmo formato que seria lido a partir do S3.

## ⚙️ Rodando no Google Colab

Este notebook lê a camada **Silver em Parquet**, particionada por `ano_pesquisa`
(pasta `Silver/state_of_data/ano_pesquisa=2023/...`, `=2024/`, `=2025-2026/`).

No Colab, a forma mais simples é:

1. Compacte a pasta `Silver/` (com as subpastas de partição) em um `.zip`.
2. Rode a célula abaixo para enviar o zip e descompactar.
3. Confirme que o `pyarrow` está instalado (já vem por padrão no Colab; se não vier,
   descomente o `pip install`).

In [1]:
# Descomente se estiver no Google Colab:
# !pip install pyarrow -q
# from google.colab import files
# uploaded = files.upload()  # selecione o arquivo Silver.zip
# import zipfile
# with zipfile.ZipFile(list(uploaded.keys())[0]) as z:
#     z.extractall(".")


In [2]:
import pandas as pd
import pyarrow.parquet as pq
import glob

CAMINHO_SILVER = "Silver/state_of_data"  # equivalente a s3://.../Silver/state_of_data/

df_silver = pd.read_parquet(CAMINHO_SILVER, engine="pyarrow")
print(f"Silver carregada: {df_silver.shape[0]} linhas x {df_silver.shape[1]} colunas")
df_silver.head()


Silver carregada: 14002 linhas x 17 colunas


,id,idade,genero,regiao,nivel_ensino,cargo_atual,senioridade,faixa_salarial,exp_dados,modelo_trabalho,linguagem_preferida,cloud_dia_a_dia,ferramenta_bi_dia_a_dia,tipo_uso_ia_generativa_empresa,uso_chatgpt_copilot,senioridade_comparavel,ano_pesquisa
0,0s7x7lc2foi894m6pogxa0s7x7ltfytz,26,Masculino,Sul,Estudante de Graduação,Analista de Dados/Data Analyst,Júnior,de R$ 2.001/mês a R$ 3.000/mês,Menos de 1 ano,Modelo 100% remoto,Python,Google Cloud (GCP),Fazemos todas as análises utilizando apenas Ex...,Colaboradores utilizando soluções baseadas em ...,Não utilizo nenhum tipo de solução de IA Gener...,Júnior,2023
1,27xq46s54lrteq2l59a27xq4uwe5fhft,32,Masculino,Sudeste,Pós-graduação,Engenheiro de Dados/Arquiteto de Dados/Data En...,Sênior,de R$ 12.001/mês a R$ 16.000/mês,de 7 a 10 anos,Modelo 100% remoto,Python,Snowflake,"Mode, Outra opção",Equipes de desenvolvimento utilizando soluções...,Não utilizo nenhum tipo de solução de IA Gener...,Sênior,2023
2,78yryvfm3j8jmw3ilr878yryz7gqhuou,22,Masculino,Sudeste,Estudante de Graduação,Analista de Dados/Data Analyst,Júnior,de R$ 2.001/mês a R$ 3.000/mês,Menos de 1 ano,Modelo 100% presencial,Python,Cloud Própria,Fazemos todas as análises utilizando apenas Ex...,Colaboradores utilizando soluções baseadas em ...,Utilizo apenas soluções gratuitas (como por ex...,Júnior,2023
3,8ma6xzjc593rfl7qsmet8ma6k85ss8zy,22,Feminino,Sudeste,Graduação/Bacharelado,Analista de Dados/Data Analyst,Júnior,de R$ 3.001/mês a R$ 4.000/mês,de 3 a 4 anos,Modelo 100% presencial,R,"Servidores On Premise/Não utilizamos Cloud, Or...",Microsoft PowerBI,Colaboradores utilizando soluções baseadas em ...,Utilizo apenas soluções gratuitas (como por ex...,Júnior,2023
4,hmzr7521jkepthmoyyvq6rgmz18lboz9,39,Feminino,Sudeste,Mestrado,Professor/Pesquisador,Pleno,de R$ 1.001/mês a R$ 2.000/mês,de 3 a 4 anos,Modelo 100% remoto,Python,Servidores On Premise/Não utilizamos Cloud,Microsoft PowerBI,Não sei opinar.,Utilizo apenas soluções gratuitas (como por ex...,Pleno,2023


## 1. Schema da tabela Silver

In [3]:
df_silver.dtypes.to_frame(name="tipo")


,tipo
id,str
idade,str
genero,str
regiao,str
nivel_ensino,str
cargo_atual,str
senioridade,str
faixa_salarial,str
exp_dados,str
modelo_trabalho,str


## 2. Volume por ano (particionamento)

In [4]:
df_silver["ano_pesquisa"].value_counts().sort_index()


ano_pesquisa
2023         5293
2024         5215
2025-2026    3494
Name: count, dtype: int64

**Validação:** os totais batem com o esperado após a limpeza aplicada no Glue Job
(remoção de duplicatas e de linha 100% vazia): 2023 = 5.293, 2024 = 5.215, 2025-2026 = 3.494.

## 3. Checagem de duplicatas (deveria ser zero — já tratado na Silver)

In [5]:
duplicadas = df_silver.duplicated().sum()
print(f"Linhas duplicadas na Silver: {duplicadas}")
assert duplicadas == 0, "Não deveria haver duplicatas na Silver!"
print("OK - nenhuma duplicata encontrada, como esperado.")


Linhas duplicadas na Silver: 0
OK - nenhuma duplicata encontrada, como esperado.


## 4. Completude por coluna (% de preenchimento)

In [6]:
completude = (1 - df_silver.isna().mean()) * 100
completude.sort_values(ascending=False).round(1).to_frame(name="pct_preenchido")


,pct_preenchido
id,100.0
idade,100.0
genero,100.0
nivel_ensino,100.0
ano_pesquisa,100.0
regiao,97.2
exp_dados,91.7
modelo_trabalho,91.7
faixa_salarial,91.7
cargo_atual,72.7


**Leitura:** `id`, `idade`, `genero`, `regiao`, `nivel_ensino` e `ano_pesquisa` têm
preenchimento próximo de 100% (perguntas obrigatórias no início do formulário).
Campos como `tipo_uso_ia_generativa_empresa` e `uso_chatgpt_copilot` têm mais nulo —
consistente com serem perguntas de blocos mais específicos, respondidas por menos gente.

## 5. Validação das correções aplicadas na Silver

In [7]:
print("--- Checagem 1: erro 'R$ 3000' foi corrigido? ---")
tem_erro_3000 = df_silver["faixa_salarial"].astype(str).str.contains("R\$ 3000", na=False).sum()
print(f"Registros com o erro antigo: {tem_erro_3000} (esperado: 0)")

print()
print("--- Checagem 2: erro 'R$ 101' foi desconsiderado (virou nulo)? ---")
tem_erro_101 = df_silver["faixa_salarial"].astype(str).str.contains("R\$ 101", na=False).sum()
print(f"Registros com o erro antigo: {tem_erro_101} (esperado: 0)")

print()
print("--- Checagem 3: categoria corrigida existe? ---")
print(df_silver["faixa_salarial"].value_counts().filter(like="30.000"))


--- Checagem 1: erro 'R$ 3000' foi corrigido? ---
Registros com o erro antigo: 0 (esperado: 0)

--- Checagem 2: erro 'R$ 101' foi desconsiderado (virou nulo)? ---
Registros com o erro antigo: 0 (esperado: 0)

--- Checagem 3: categoria corrigida existe? ---
faixa_salarial
de R$ 25.001/mês a R$ 30.000/mês    422
Name: count, dtype: int64


<>:2: SyntaxWarning: invalid escape sequence '\$'
<>:7: SyntaxWarning: invalid escape sequence '\$'
<>:2: SyntaxWarning: invalid escape sequence '\$'
<>:7: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_142/2573765922.py:2: SyntaxWarning: invalid escape sequence '\$'
  tem_erro_3000 = df_silver["faixa_salarial"].astype(str).str.contains("R\$ 3000", na=False).sum()
/tmp/ipykernel_142/2573765922.py:7: SyntaxWarning: invalid escape sequence '\$'
  tem_erro_101 = df_silver["faixa_salarial"].astype(str).str.contains("R\$ 101", na=False).sum()


## 6. Distribuição de senioridade (original vs. comparável)

In [8]:
comparacao = pd.crosstab(df_silver["senioridade"], df_silver["senioridade_comparavel"], dropna=False)
comparacao


senioridade_comparavel,Júnior,Pleno,Sênior,NaN
senioridade,,,,
Especialista/Staff+,0,0,349,0
Júnior,2432,0,0,0
Pleno,0,3543,0,0
Sênior,0,0,3849,0
NaN,0,0,0,3829


**Validação:** confirma que todo registro "Especialista/Staff+" (categoria só de
2025-2026) foi agrupado em "Sênior" na coluna `senioridade_comparavel`, sem alterar a
coluna original `senioridade`.

## 7. Amostra de dados por ano

In [9]:
for ano in sorted(df_silver["ano_pesquisa"].unique()):
    print(f"--- Amostra {ano} ---")
    display(df_silver[df_silver["ano_pesquisa"] == ano][
        ["idade","genero","regiao","cargo_atual","senioridade","faixa_salarial","modelo_trabalho"]
    ].head(3))
    print()


--- Amostra 2023 ---


,idade,genero,regiao,cargo_atual,senioridade,faixa_salarial,modelo_trabalho
0,26,Masculino,Sul,Analista de Dados/Data Analyst,Júnior,de R$ 2.001/mês a R$ 3.000/mês,Modelo 100% remoto
1,32,Masculino,Sudeste,Engenheiro de Dados/Arquiteto de Dados/Data En...,Sênior,de R$ 12.001/mês a R$ 16.000/mês,Modelo 100% remoto
2,22,Masculino,Sudeste,Analista de Dados/Data Analyst,Júnior,de R$ 2.001/mês a R$ 3.000/mês,Modelo 100% presencial



--- Amostra 2024 ---


,idade,genero,regiao,cargo_atual,senioridade,faixa_salarial,modelo_trabalho
5293,19,Masculino,Centro-oeste,Outra Opção,Júnior,Menos de R$ 1.000/mês,Modelo 100% presencial
5294,23,Masculino,Sudeste,Cientista de Dados/Data Scientist,Júnior,de R$ 6.001/mês a R$ 8.000/mês,Modelo híbrido flexível (o funcionário tem lib...
5295,26,Masculino,Sudeste,Analytics Engineer,Sênior,de R$ 12.001/mês a R$ 16.000/mês,Modelo 100% remoto



--- Amostra 2025-2026 ---


,idade,genero,regiao,cargo_atual,senioridade,faixa_salarial,modelo_trabalho
10508,28,Feminino,Nordeste,Analista de BI/BI Analyst,Júnior,de R$ 4.001/mês a R$ 6.000/mês,Modelo híbrido com dias fixos de trabalho pres...
10509,29,Feminino,Sudeste,Analista de Negócios/Business Analyst,Sênior,de R$ 8.001/mês a R$ 12.000/mês,Modelo 100% presencial
10510,31,Masculino,Sudeste,Analista de Dados/Data Analyst,Sênior,de R$ 20.001/mês a R$ 25.000/mês,Modelo 100% remoto


## 8. Resumo executivo desta validação

In [10]:
print(f"Total de linhas na Silver: {len(df_silver)}")
print(f"Total de colunas: {len(df_silver.columns)}")
print(f"Anos presentes: {sorted(df_silver['ano_pesquisa'].unique())}")
print(f"Duplicatas: {df_silver.duplicated().sum()} (esperado: 0)")
print(f"Erros de digitação conhecidos ainda presentes: {tem_erro_3000 + tem_erro_101} (esperado: 0)")
print()
print("Conclusão: a camada Silver está íntegra e pronta para alimentar a camada Gold.")


Total de linhas na Silver: 14002
Total de colunas: 17
Anos presentes: ['2023', '2024', '2025-2026']
Duplicatas: 0 (esperado: 0)
Erros de digitação conhecidos ainda presentes: 0 (esperado: 0)

Conclusão: a camada Silver está íntegra e pronta para alimentar a camada Gold.
